# Evaluation with Pydantic Evals

Pydantic Evals is a framework for systematically testing and evaluating AI systems. It provides a code-first approach where all evaluation components are defined in Python.

```
Dataset (1) ──────────── (Many) Case
│                        │
│                        │
└─── (Many) Experiment ──┴─── (Many) Case results
     └─── (1) Task
     └─── (Many) Evaluator
```

Key concepts:
- **Dataset**: A collection of test Cases
- **Case**: A single test scenario with inputs, expected outputs, and metadata
- **Evaluator**: Analyzes and scores task results (deterministic or LLM-based)

In [ ]:
import nest_asyncio

nest_asyncio.apply()

## Imports

In [ ]:
from typing import Any

import logfire
from datasets import load_dataset
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from pydantic_evals import Case, Dataset
from pydantic_evals.evaluators import Evaluator, EvaluatorContext, IsInstance

load_dotenv()

logfire.configure()
logfire.instrument_pydantic_ai()

## Creating the evaluation dataset

In [ ]:
ds = load_dataset("AI-MO/aimo-validation-aime")
ds["train"][0]

In [ ]:
cases = [
    Case(
        name=f"aime_{i}",
        inputs=d["problem"],
        expected_output=int(d["answer"]),
        metadata={"source": "AIME"},
    )
    for i, d in enumerate(ds["train"])
][:10]

print(f"Created {len(cases)} cases")
print(f"Example case: {cases[0].name}")
print(f"Input: {cases[0].inputs[:100]}...")
print(f"Expected output: {cases[0].expected_output}")

## Defining the agent

In [ ]:
class MathResponse(BaseModel):
    explanation: str = Field(description="The explanation of the answer")
    answer: int = Field(
        description="The answer to the question. It should be an integer."
    )


math_agent = Agent(
    "openai:gpt-5-nano",
    system_prompt="You're a math expert. Solve the given problem step by step.",
    output_type=MathResponse,
)

## Defining evaluators

In [ ]:
class AnswerAccuracy(Evaluator[str, MathResponse, Any]):
    """Check if the agent's answer matches the expected answer."""

    def evaluate(self, ctx: EvaluatorContext[str, MathResponse, Any]) -> bool:
        return ctx.output.answer == ctx.expected_output

## Building the dataset and running the evaluation

In [ ]:
async def solve_math(question: str) -> MathResponse:
    result = await math_agent.run(question)
    return result.output


dataset = Dataset(
    cases=cases,
    evaluators=[
        IsInstance(type_name="MathResponse"),
        AnswerAccuracy(),
    ],
)

In [ ]:
report = await dataset.evaluate(solve_math)
report.print(include_input=True, include_output=True)

# Exercise

Create an LLM judge that evaluates the accuracy of the answer and the clarity of its explanation.

Use the `LastLetterConcat` dataset where the task is to concatenate the last letters of given words.

In [ ]:
ds = load_dataset("ChilleD/LastLetterConcat")

cases = [
    Case(
        name=f"lastletter_{i}",
        inputs=d["question"],
        expected_output=d["answer"],
        metadata={"source": "LastLetterConcat"},
    )
    for i, d in enumerate(ds["train"])
][:20]

print(f"Created {len(cases)} cases")
print(f"Example: {cases[0].inputs} -> {cases[0].expected_output}")

In [ ]:
class PuzzleResponse(BaseModel):
    explanation: str = Field(description="The explanation of the answer")
    answer: str = Field(
        description="The answer to the question. It should be a string with 4 characters.",
        pattern=r"^[a-zA-Z]{4}$",
    )


puzzle_agent = Agent(
    "openai:gpt-5-nano",
    system_prompt="You're a puzzle expert. Solve the given problem step by step.",
    output_type=PuzzleResponse,
)

In [ ]:
class ClarityScore(BaseModel):
    reason: str = Field(description="Brief justification for the score")
    score: float = Field(description="Clarity score from 0 to 1", ge=0, le=1)


class AccuracyResult(BaseModel):
    reason: str = Field(description="Brief justification for the result")
    is_accurate: bool = Field(
        description="Whether the answer correctly concatenates the last letters"
    )


clarity_judge = Agent(
    "openai:gpt-5-nano",
    system_prompt=(
        "You evaluate whether an explanation clearly and logically describes "
        "how to arrive at an answer, step by step. "
        "Score from 0 (incoherent) to 1 (perfectly clear)."
    ),
    output_type=ClarityScore,
)

accuracy_judge = Agent(
    "openai:gpt-5-nano",
    system_prompt=(
        "You evaluate whether an answer correctly concatenates the last letters "
        "of each word in the input. Compare the output to the expected output."
    ),
    output_type=AccuracyResult,
)


class ClarityEvaluator(Evaluator[str, PuzzleResponse, Any]):
    """Evaluate clarity of the explanation using an LLM."""

    async def evaluate(self, ctx: EvaluatorContext[str, PuzzleResponse, Any]) -> float:
        prompt = (
            f"Input: {ctx.inputs}\n"
            f"Explanation: {ctx.output.explanation}\n"
            f"Answer: {ctx.output.answer}"
        )
        result = await clarity_judge.run(prompt)
        return result.output.score


class AccuracyEvaluator(Evaluator[str, PuzzleResponse, Any]):
    """Evaluate answer accuracy using an LLM."""

    async def evaluate(self, ctx: EvaluatorContext[str, PuzzleResponse, Any]) -> bool:
        prompt = (
            f"Input: {ctx.inputs}\n"
            f"Answer: {ctx.output.answer}\n"
            f"Expected: {ctx.expected_output}"
        )
        result = await accuracy_judge.run(prompt)
        return result.output.is_accurate


class AnswerMatch(Evaluator[str, PuzzleResponse, Any]):
    """Check if the agent's answer matches the expected answer."""

    def evaluate(self, ctx: EvaluatorContext[str, PuzzleResponse, Any]) -> bool:
        return ctx.output.answer == ctx.expected_output


async def solve_puzzle(question: str) -> PuzzleResponse:
    result = await puzzle_agent.run(question)
    return result.output


puzzle_dataset = Dataset(
    cases=cases,
    evaluators=[
        IsInstance(type_name="PuzzleResponse"),
        AnswerMatch(),
        ClarityEvaluator(),
        AccuracyEvaluator(),
    ],
)

In [ ]:
report = await puzzle_dataset.evaluate(solve_puzzle)
report.print(include_input=True, include_output=True, include_reasons=True)